# Winner Side Prediction Baseline

Train a Random Forest model to predict whether the winner is represented as `R` or `B` using pre-fight career stats.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline


RANDOM_STATE = 42
TEST_SIZE = 0.15
CV_FOLDS = 5
N_RANDOM_SPLITS = 5

data_path = Path("../data_clean/pre_fight_career_stats.csv")
df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Keep only fights with a red-side or blue-side winner.
df = df[df["winner_side"].isin(["R", "B"])].copy()
df = df.dropna(subset=["date", "winner_side"]).reset_index(drop=True)

print("Loaded:", data_path)
print("Shape:", df.shape)
print(df["winner_side"].value_counts())
df.head()

Loaded: ..\data_clean\pre_fight_career_stats.csv
Shape: (3040, 121)
winner_side
R    1825
B    1215
Name: count, dtype: int64


,fight_id,date,R_fighter,B_fighter,R_age,B_age,winner,winner_side,win_by,R_elo,...,B_DISTANCE_absorbed_per_min,B_DISTANCE_opp_att_per_min,B_CLINCH_land_per_min,B_CLINCH_att_per_min,B_CLINCH_absorbed_per_min,B_CLINCH_opp_att_per_min,B_GROUND_land_per_min,B_GROUND_att_per_min,B_GROUND_absorbed_per_min,B_GROUND_opp_att_per_min
0,8644,1995-12-16,Dan Severn,Oleg Taktarov,37.522245,28.306639,Dan Severn,R,Decision - Unanimous,1573.348793,...,0.581583,2.132472,0.038772,0.135703,0.213247,0.465267,0.000000,0.000000,0.678514,1.124394
1,8645,1995-12-16,Oleg Taktarov,Dave Beneteau,28.306639,NaN,Oleg Taktarov,R,Submission,1558.827392,...,0.000000,0.000000,0.920716,2.455243,0.153453,0.613811,3.222506,5.217391,0.153453,0.153453
2,8647,1995-12-16,Dan Severn,Paul Varelans,37.522245,26.245038,Dan Severn,R,Submission,1593.683396,...,1.912568,3.606557,0.491803,0.765027,0.327869,1.366120,0.327869,0.327869,0.928962,1.475410
3,8628,1996-05-17,Dan Severn,Ken Shamrock,37.941136,32.262834,Dan Severn,R,Decision - Split,1609.448071,...,0.205781,0.911318,0.235179,0.293974,0.146987,0.235179,1.028907,1.499265,0.146987,0.205781
4,8606,1996-12-07,Ken Shamrock,Brian Johnston,32.821355,28.361396,Ken Shamrock,R,Submission,1562.685802,...,0.203390,0.508475,1.220339,1.830508,0.508475,0.711864,0.711864,1.118644,1.627119,1.830508


## Balance Side Labels

Each fight can be represented in either orientation. If a fight is swapped, every matching `R_...` and `B_...` column changes sides, `R_fighter` and `B_fighter` change sides, and `winner_side` flips. This balances the label without duplicating fights.

In [3]:
def find_red_blue_pairs(columns):
    pairs = []
    for col in columns:
        if col.startswith("R_"):
            b_col = "B_" + col[2:]
            if b_col in columns:
                pairs.append((col, b_col))
    return pairs


def swap_red_blue_rows(data, swap_mask):
    balanced = data.copy()
    pairs = find_red_blue_pairs(balanced.columns)
    r_cols = [r for r, _ in pairs]
    b_cols = [b for _, b in pairs]

    r_values = balanced.loc[swap_mask, r_cols].to_numpy(copy=True)
    b_values = balanced.loc[swap_mask, b_cols].to_numpy(copy=True)
    balanced.loc[swap_mask, r_cols] = b_values
    balanced.loc[swap_mask, b_cols] = r_values

    balanced.loc[swap_mask, "winner_side"] = balanced.loc[swap_mask, "winner_side"].map({"R": "B", "B": "R"})

    return balanced


def randomly_balance_winner_side(data, random_state=42, balance_by_year=True):
    rng = np.random.default_rng(random_state)
    balanced = data.copy().reset_index(drop=True)

    desired_side = pd.Series(index=balanced.index, dtype="object")
    groups = balanced["date"].dt.year if balance_by_year else pd.Series(0, index=balanced.index)

    for _, group_idx in balanced.groupby(groups).groups.items():
        group_idx = np.array(list(group_idx))
        n_rows = len(group_idx)
        group_sides = ["R"] * (n_rows // 2) + ["B"] * (n_rows // 2)

        if n_rows % 2 == 1:
            group_sides.append(rng.choice(["R", "B"]))

        group_sides = np.array(group_sides)
        rng.shuffle(group_sides)
        desired_side.loc[group_idx] = group_sides

    swap_mask = balanced["winner_side"].to_numpy() != desired_side.to_numpy()
    balanced = swap_red_blue_rows(balanced, swap_mask)
    balanced["target"] = (balanced["winner_side"] == "R").astype(int)

    return balanced


balanced_df = randomly_balance_winner_side(df, random_state=RANDOM_STATE)

print("Original target distribution:")
print(df["winner_side"].value_counts())
print("\nBalanced target distribution:")
print(balanced_df["winner_side"].value_counts())

balanced_df.sort_values("date").groupby(balanced_df["date"].dt.year)["winner_side"].value_counts(normalize=True).unstack().tail(10)

Original target distribution:
winner_side
R    1825
B    1215
Name: count, dtype: int64

Balanced target distribution:
winner_side
R    1523
B    1517
Name: count, dtype: int64


winner_side,B,R
date,,
2017,0.497110,0.502890
2018,0.500000,0.500000
2019,0.497207,0.502793
2020,0.502924,0.497076
2021,0.497608,0.502392
2022,0.500000,0.500000
2023,0.500000,0.500000
2024,0.500000,0.500000
2025,0.502075,0.497925


## Build Model Features

Outcome and identity columns are excluded from features. Matching `R_...` and `B_...` stats are also turned into difference features.

In [4]:
def is_logdiff_candidate(suffix):
    skip_terms = ["pct", "acc", "def", "quickness", "elo", "age"]
    return not any(term in suffix.lower() for term in skip_terms)


def is_centered_sq_candidate(suffix):
    skip_terms = ["pct", "acc", "def", "quickness", "elo"]
    return not any(term in suffix.lower() for term in skip_terms)


def add_matchup_features(data):
    features = data.copy()
    for r_col, b_col in find_red_blue_pairs(features.columns):
        suffix = r_col[2:]
        if pd.api.types.is_numeric_dtype(features[r_col]) and pd.api.types.is_numeric_dtype(features[b_col]):
            features[f"DIFF_{suffix}"] = features[r_col] - features[b_col]

            if is_centered_sq_candidate(suffix):
                center = pd.concat([features[r_col], features[b_col]], ignore_index=True).median()
                features[f"DIFF_CENTERED_SQ_{suffix}"] = (
                    (features[r_col] - center) ** 2
                    - (features[b_col] - center) ** 2
                )

            if is_logdiff_candidate(suffix):
                r_values = features[r_col].clip(lower=0)
                b_values = features[b_col].clip(lower=0)
                features[f"LOGDIFF_{suffix}"] = np.log1p(r_values) - np.log1p(b_values)
    return features


model_df = add_matchup_features(balanced_df)

leakage_cols = [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "R_opponent",
    "B_opponent",
    "winner",
    "winner_side",
    "R_result",
    "B_result",
    "R_won",
    "B_won",
    "win_by",
    "target",
]

candidate_cols = [col for col in model_df.columns if col not in leakage_cols]
feature_cols = [col for col in candidate_cols if pd.api.types.is_numeric_dtype(model_df[col])]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df["target"]

print("Rows:", len(model_df))
print("Features:", len(feature_cols))
feature_cols[:10]

Rows: 3040
Features: 254


C:\Users\Abtin\AppData\Local\Temp\ipykernel_19900\1793784410.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"LOGDIFF_{suffix}"] = np.log1p(r_values) - np.log1p(b_values)
C:\Users\Abtin\AppData\Local\Temp\ipykernel_19900\1793784410.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  features[f"DIFF_{suffix}"] = features[r_col] - features[b_col]
C:\Users\Abtin\AppData\Local\Temp\ipykernel_19900\1793784410.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.inser

['R_age',
 'B_age',
 'R_elo',
 'B_elo',
 'R_total_fights',
 'R_total_fight_time',
 'R_fight_quickness',
 'R_win_DEC_pct',
 'R_win_KO_TKO_pct',
 'R_win_SUB_pct']

## Feature Set Comparison

Compare feature groups with five random stratified 90/10 splits.

In [5]:
def make_rf_pipeline(random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=600,
                    min_samples_leaf=8,
                    max_features="sqrt",
                    n_jobs=-1,
                    random_state=random_state,
                    class_weight=None,
                ),
            ),
        ]
    )


all_numeric_feature_cols = feature_cols.copy()
diff_only_cols = [
    col for col in all_numeric_feature_cols
    if col.startswith("DIFF_") and not col.startswith("DIFF_CENTERED_SQ_")
]
logdiff_cols = [col for col in all_numeric_feature_cols if col.startswith("LOGDIFF_")]
centered_sq_cols = [col for col in all_numeric_feature_cols if col.startswith("DIFF_CENTERED_SQ_")]
raw_rb_cols = [
    col for col in all_numeric_feature_cols
    if col.startswith(("R_", "B_"))
]

logdiff_suffixes = {col.replace("LOGDIFF_", "") for col in logdiff_cols}
logdiff_when_applicable_cols = []
for col in diff_only_cols:
    suffix = col.replace("DIFF_", "")
    logdiff_col = f"LOGDIFF_{suffix}"
    if suffix in logdiff_suffixes and logdiff_col in all_numeric_feature_cols:
        logdiff_when_applicable_cols.append(logdiff_col)
    else:
        logdiff_when_applicable_cols.append(col)

centered_sq_suffixes = {col.replace("DIFF_CENTERED_SQ_", "") for col in centered_sq_cols}
centered_sq_when_applicable_cols = []
for col in diff_only_cols:
    suffix = col.replace("DIFF_", "")
    centered_sq_col = f"DIFF_CENTERED_SQ_{suffix}"
    if suffix in centered_sq_suffixes and centered_sq_col in all_numeric_feature_cols:
        centered_sq_when_applicable_cols.append(centered_sq_col)
    else:
        centered_sq_when_applicable_cols.append(col)

feature_sets = {
    "DIFF-only features": diff_only_cols,
    "LOGDIFF when applicable": logdiff_when_applicable_cols,
    "CENTERED_SQ when applicable": centered_sq_when_applicable_cols,
    "R/B raw stats only": raw_rb_cols,
}


def evaluate_feature_set(name, cols):
    X_subset = model_df[cols].replace([np.inf, -np.inf], np.nan)
    splitter = StratifiedShuffleSplit(
        n_splits=N_RANDOM_SPLITS,
        test_size=0.10,
        random_state=RANDOM_STATE,
    )

    accuracies = []
    for split_num, (train_idx, test_idx) in enumerate(splitter.split(X_subset, y), start=1):
        model = make_rf_pipeline(RANDOM_STATE + split_num)
        model.fit(X_subset.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X_subset.iloc[test_idx])
        accuracies.append(accuracy_score(y.iloc[test_idx], pred))

    return {
        "feature_set": name,
        "n_features": len(cols),
        "mean_accuracy": np.mean(accuracies),
        "std_accuracy": np.std(accuracies),
        "split_accuracies": accuracies,
    }


feature_set_results = pd.DataFrame(
    evaluate_feature_set(name, cols)
    for name, cols in feature_sets.items()
).sort_values("mean_accuracy", ascending=False).reset_index(drop=True)

selected_feature_set_name = feature_set_results.loc[0, "feature_set"]
feature_cols = feature_sets[selected_feature_set_name]
X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)

print("Best feature set:", selected_feature_set_name)
print("Selected feature count:", len(feature_cols))
feature_set_results

Best feature set: DIFF-only features
Selected feature count: 57


,feature_set,n_features,mean_accuracy,std_accuracy,split_accuracies
0,DIFF-only features,57,0.635526,0.032855,"[0.5888157894736842, 0.625, 0.631578947368421,..."
1,LOGDIFF when applicable,57,0.617763,0.035938,"[0.5855263157894737, 0.6052631578947368, 0.582..."
2,R/B raw stats only,114,0.612500,0.019781,"[0.5986842105263158, 0.6052631578947368, 0.598..."
3,CENTERED_SQ when applicable,57,0.580921,0.038844,"[0.555921052631579, 0.5493421052631579, 0.5592..."


## Random Train/Test Split

The split is random and stratified, so the train and test sets keep the same balanced target mix.

In [6]:
model_df = model_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).rename({1: "R", 0: "B"}))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).rename({1: "R", 0: "B"}))

Train shape: (2584, 57)
Test shape: (456, 57)

Train target distribution:
target
R    0.501161
B    0.498839
Name: proportion, dtype: float64

Test target distribution:
target
B    0.5
R    0.5
Name: proportion, dtype: float64


In [7]:
rf_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=600,
                min_samples_leaf=8,
                max_features="sqrt",
                n_jobs=-1,
                random_state=RANDOM_STATE,
                class_weight=None,
            ),
        ),
    ]
)

rf_model.fit(X_train, y_train)

pred = rf_model.predict(X_test)
proba = rf_model.predict_proba(X_test)[:, 1]

print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred))

importances = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": rf_model.named_steps["model"].feature_importances_,
    }
).sort_values("importance", ascending=False)

importances.head(25)


Confusion matrix:
[[125 103]
 [ 76 152]]


,feature,importance
0,DIFF_age,0.062533
1,DIFF_elo,0.035950
25,DIFF_TOTAL_STR_absorbed_per_min,0.029501
35,DIFF_HEAD_absorbed_per_min,0.027349
56,DIFF_GROUND_opp_att_per_min,0.022973
55,DIFF_GROUND_absorbed_per_min,0.022415
22,DIFF_SIG_STR_def,0.022024
33,DIFF_HEAD_land_per_min,0.020290
12,DIFF_SIG_STR_acc,0.020283
23,DIFF_TOTAL_STR_land_per_min,0.020066
